# Group project: which brain cells does Parkinson's disease hit first?

**ISN Neurochemistry School 2026**

Parkinson's disease is defined by the loss of dopaminergic neurons. But patients
notice something else first: about a decade before any tremor, most of them lose
their sense of smell. Nobody is sure why, and nobody knows which cells are in
trouble before the dying starts, because by the time a patient reaches a clinic
it has been going on for years.

Flies let you look early. Pech et al. (*eLife*, 2025) built five fly models of
familial Parkinsonism, one for each of five human disease genes, and sequenced
every cell in the brains of **young** flies, before anything has degenerated.
For every cell type they measured how far its gene expression had drifted from
normal.

## Your group has one of the five models

Each group takes one disease gene and answers the same four questions about it.
Nobody can answer the real question alone: whether the five genes converge on
the same cells, or each break something different. That only appears when all
five groups have presented.

| model | human gene | what the protein normally does |
|---|---|---|
| `LRRK2G2019S` | *LRRK2* | a kinase; the commonest genetic cause of Parkinson's |
| `PINK1P399L` | *PINK1* | tags damaged mitochondria so the cell can clear them |
| `Rab39BG192R` | *RAB39B* | moves membrane cargo around the cell |
| `AuxR927G` | *DNAJC6*, auxilin | recycles vesicles at the synapse |
| `SynjR258Q` | *SYNJ1*, synaptojanin | recycles vesicles at the synapse |

**Your four questions**

1. Which cell types does **your** model put into trouble?
2. Do the dopaminergic neurons, the ones that die in the disease, show anything?
3. Which genes change in **your** model?
4. Are those genes special to the affected cells, or does every neuron have them?

By the end you will have filled in a results box and have two figures. Three
slides come straight out of it.

Everything is `pandas` and `scanpy`. No model, no training. Allow about 30
minutes to run, then spend your time arguing about what it means.

## Running on Google Colab

Run the cell below first. On Colab it installs the software and fetches the
course files; on a laptop it does nothing. Run it either way.

In [ ]:
# Colab setup. On a laptop install this cell does nothing.
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import os

    !pip install -q scanpy openpyxl "pandas==2.2.3"

    # Get the course files, or update them if this session already has them.
    import subprocess

    if os.path.isdir("/content/course/.git"):
        update = subprocess.run(
            ["git", "-C", "/content/course", "pull", "--quiet", "--ff-only"],
            capture_output=True, text=True,
        )
        if update.returncode:
            print("Could not update the course files. Continuing with the copy")
            print("already in this session, which is almost certainly fine.")
    else:
        !rm -rf /content/course
        !git clone -q https://github.com/nkempynck/isn-school-2026 /content/course
    %cd /content/course
    print("Ready.")
else:
    print("Running locally, nothing to install.")

Running locally, nothing to install.


## 0. Setup, and your group's model

Put your group's model in `MY_MODEL`. Everything below follows from it, so this
is the only line you have to change to make this notebook yours.

In [ ]:
MY_MODEL = "AuxR927G"     # <-- YOUR GROUP'S MODEL. Change this first.

MODELS = ["LRRK2G2019S", "PINK1P399L", "Rab39BG192R", "AuxR927G", "SynjR258Q"]
assert MY_MODEL in MODELS, f"pick one of {MODELS}"

import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.set_figure_params(figsize=(6, 5), dpi=90, frameon=False)
print("working on:", MY_MODEL)

working on: AuxR927G


## 1. The data

For every cell type in the brain the authors worked out how much change in gene
expression you would **expect**, given how many cells of that type they
captured, then measured how far the real data missed that expectation.

That miss is the number in the table.

- **Large and positive**: this cell type is far more deregulated than its size
  explains. This is what we are looking for.
- **Around zero**: nothing unusual.
- **Large and negative**: it changed *less* than expected. This happens to the
  very large populations and is a property of the measure, not a finding.

In [ ]:
from scripts.download_data import pech_tables

paths = pech_tables()

cells = pd.read_excel(paths["celltypes"], sheet_name="normalized residuals z_scores")
cells = (cells[["cell type"] + MODELS]
         .dropna(subset=["cell type"])
         .set_index("cell type")
         .apply(pd.to_numeric, errors="coerce")
         .dropna(how="all"))

# Not every cluster was given a cell type name; the rest are still numbered.
named = cells[[not str(i).isdigit() for i in cells.index]]

print(f"\n{len(cells)} clusters, of which {len(named)} have a cell type name")
cells.head()

Published spreadsheets are laid out for human eyes, not for `pandas`. This one
has blank spacer columns and summary columns mixed in with the data, which is
why the cell above picks its columns by name instead of taking the sheet as it
comes. Expect to do this every time you use a supplementary table.

## 2. Question 1: which cell types does your model put into trouble?

Run this, then look at the figure before reading anything else.

In [ ]:
mine = named[MY_MODEL].sort_values(ascending=False).head(20).sort_values()

fig, ax = plt.subplots(figsize=(7.2, 0.3 * len(mine) + 1.3))
ax.barh(range(len(mine)), mine.values,
        color=["#c0392b" if v > 0 else "#7d9bc1" for v in mine.values])
ax.set_yticks(range(len(mine)))
ax.set_yticklabels(mine.index, fontsize=9)
ax.axvline(0, color="#333333", lw=0.8)
ax.grid(False)
ax.set_xlabel("deregulation, further right means further from normal")
ax.set_title(f"{MY_MODEL}: the twenty cell types that drift furthest", fontsize=11)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.show()

named[MY_MODEL].sort_values(ascending=False).head(5).round(1)

Answer these before moving on. Write the answers down; they go on your slides.

- Which cell type is top for your model, and how far clear of the second is it?
- **What is that cell type and what does it do?** If the abbreviation means
  nothing to you, look it up now. It is your whole result.
- Does your model have one standout, or a handful of similar ones?

## 3. Question 2: what about the dopaminergic neurons?

Parkinson's disease kills dopaminergic neurons. This brain has them, annotated
as central brain dopaminergic neurons, `CB DAN`.

Compare them with your top cell type.

In [ ]:
WATCH = ["CB DAN", named[MY_MODEL].idxmax()]

cells.loc[WATCH, MODELS].round(1)

- What is the dopaminergic number in **your** model?
- Put it next to your top cell type. What do you conclude, and what can you
  *not* conclude from flies this young?

## 4. Is your result yours alone, or do the others agree?

You cannot answer this properly until the other groups present. But you can see
the shape of it now, which will let you ask them the right question on Friday.

In [ ]:
top = named.loc[named[MY_MODEL].sort_values(ascending=False).head(10).index]

fig, ax = plt.subplots(figsize=(7, 4.4))
sns.heatmap(top, cmap="RdBu_r", center=0, vmin=-200, vmax=200,
            annot=True, fmt=".0f", annot_kws={"size": 7},
            cbar_kws={"label": "deregulation"}, ax=ax)
ax.set_title(f"Your top ten cell types, in all five models", fontsize=11)
ax.set_ylabel("")
ax.grid(False)
plt.show()

# How similarly do the five models rank the cell types?
named.corr(method="spearman")[MY_MODEL].round(2).sort_values(ascending=False)

## 5. Question 3: which genes change in your model?

The authors ranked every fly gene by how deregulated it is, separately for each
model. **Rank 0 is the most deregulated**, so small numbers matter.

In [ ]:
genes = pd.read_excel(paths["genes"], sheet_name="Ranked_Genes_Fly")

# The gene names are in a column pandas calls "Unnamed: 10", because
# "sorted by median" in the sheet is a heading over the block, not a column.
cols = ["Unnamed: 10"] + [m + ".1" for m in MODELS] + ["median rank"]
genes = genes[cols].dropna(subset=["Unnamed: 10"])
genes.columns = ["gene"] + MODELS + ["median_rank"]
genes = genes.set_index("gene").apply(pd.to_numeric, errors="coerce").dropna()

print(f"{len(genes):,} genes ranked\n")
genes[MY_MODEL].nsmallest(15).astype(int)

Look up several of your top genes. Do they have anything in common? Does it fit
what your disease gene normally does, from the table at the top of this
notebook?

Be sceptical as well. Ribosomal RNA, mitochondrial transcripts and histone genes
turn up near the top of a lot of rankings for technical reasons. If your list is
full of them, say so: that is a finding about the experiment.

## 6. Question 4: are those genes special to the affected cells?

If your model's genes were only made by the cell type that got into trouble,
that would be a tidy explanation. Check it in the Fly Cell Atlas, the same file
you used in Practical 5.

In [ ]:
from scripts.prepare_data import make_fca_brain_h5ad

make_fca_brain_h5ad()          # 44 MB, from the course mirror
adata = sc.read_h5ad(ROOT / "data" / "processed" / "fca_brain.h5ad")

# The atlas names cell types by anatomy. Olfactory projection neurons are
# "antennal lobe projection neuron" here.
TYPES = ["antennal lobe projection neuron", "dopaminergic PAM neuron",
         "gamma Kenyon cell", "columnar neuron T1", "ensheathing glial cell"]

# Your model's top genes, keeping the ones the atlas measured.
MY_GENES = [g for g in genes[MY_MODEL].nsmallest(25).index if g in adata.var_names][:8]
print("showing:", MY_GENES)

sc.pl.dotplot(adata[adata.obs["annotation"].isin(TYPES)],
              var_names=MY_GENES, groupby="annotation")

- Are your genes specific to the cell type that was affected, or general?
- If every neuron makes them, why would one cell type fail first? Write down
  your best guess. You will not be able to prove it here, and saying so is part
  of the answer.

## 7. Your results

Run this. It prints what goes on your slides.

In [ ]:
mine_cells = named[MY_MODEL].sort_values(ascending=False)
mine_genes = genes[MY_MODEL].nsmallest(8).index.tolist()
agree = named.corr(method="spearman")[MY_MODEL].drop(MY_MODEL).sort_values(ascending=False)

print(f"MODEL                 {MY_MODEL}")
print(f"most affected         {mine_cells.index[0]}  ({mine_cells.iloc[0]:.0f})")
print(f"runner-up             {mine_cells.index[1]}  ({mine_cells.iloc[1]:.0f})")
print(f"dopaminergic neurons  CB DAN  ({cells.loc['CB DAN', MY_MODEL]:.1f})")
print(f"top genes             {', '.join(mine_genes)}")
print(f"agrees most with      {agree.index[0]}  (rho {agree.iloc[0]:.2f})")
print(f"agrees least with     {agree.index[-1]}  (rho {agree.iloc[-1]:.2f})")

## What to present

Three slides, about five minutes.

1. **Your gene.** Which human Parkinson's gene you had, what the protein
   normally does, and what the experiment was: young flies, whole brain, before
   anything has degenerated.
2. **Your answer.** Your figure from section 2, with your top cell type and the
   dopaminergic number beside it. Then your genes from section 5 and what they
   have in common.
3. **How much you believe it.** What you cannot conclude from flies this young.
   Whether your gene list looks like biology or like technical noise. And one
   question you want to ask the other four groups.

That last one matters. None of you can see the real answer alone. Come with a
question for the groups whose gene was different from yours.